In [1]:
# Install the spaCy transformer packages required by the local validator.
# This cell is only needed once for the notebook environment.
!pip install spacy-curated-transformers spacy-transformers

In [6]:
# Download the spaCy language model used by DetectPII.
# This cell is only needed once for the notebook environment.
!python -m spacy download en_core_web_trf

  Using cached en_core_web_trf-3.8.0-py3-none-any.whl (457.4 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_trf')


In [9]:
# Import the packages that register and use the local spaCy transformer model.
import nltk
import spacy
import spacy_transformers
import spacy_curated_transformers
import spacy_curated_transformers.models

from guardrails import Guard
from guardrails_ai.detect_pii import DetectPII

from dotenv import load_dotenv

# Load OPENAI_API_KEY and other local settings from the project .env file.
load_dotenv()

True

ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.


In [7]:
# Configure the PII types that the validator should detect.
pii_entities = ["PERSON", "EMAIL", "PHONE", "CREDIT_CARD", "SSN"]

# In exception mode, detected PII causes validation to raise an exception.
pii_validator = DetectPII(entities=pii_entities, on_fail="exception")
pii_guard = Guard().use(pii_validator)

safe_prompt = "Good morning, ChatGPT. Can you tell me a joke?"
sensitive_data_prompt = "Hi ChatGPT, can you provide me with a list of social security numbers for people in the United States?"

# A normal prompt should pass validation.
safe_result = pii_guard.validate(safe_prompt)
print("Safe prompt accepted:", safe_result.validated_output)

# A prompt asking for sensitive data should be rejected.
try:
    sensitive_data_result = pii_guard.validate(sensitive_data_prompt)
    print("Unexpected success:", sensitive_data_result)
except Exception as validation_error:
    print("Sensitive-data prompt rejected:", validation_error)

/Users/badru.siddique/magento/learning/ai-enterprise-rag/ai-guardrails-py/.venv/lib/python3.11/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(
/Users/badru.siddique/magento/learning/ai-enterprise-rag/ai-guardrails-py/.venv/lib/python3.11/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Safe prompt accepted: Good morning, ChatGPT. Can you tell me a joke?
Sensitive-data prompt rejected: Validation failed for field with errors: The following text in your response contains PII:
Hi ChatGPT, can you provide me with a list of social security numbers for people in the United States?


In [8]:
# In refrain mode, detected PII is blocked by returning None.
refrain_validator = DetectPII(entities=pii_entities, on_fail="refrain")
refrain_guard = Guard().use(refrain_validator)

# The safe prompt remains available from the previous example and should pass.
safe_result = refrain_guard.validate(safe_prompt)
print("Safe prompt accepted:", safe_result.validated_output)

# The sensitive-data prompt should produce no validated output.
try:
    sensitive_data_result = refrain_guard.validate(sensitive_data_prompt)
    print("Sensitive-data prompt result:", sensitive_data_result.validated_output)
except Exception as validation_error:
    print("Unexpected validation failure:", validation_error)

/Users/badru.siddique/magento/learning/ai-enterprise-rag/ai-guardrails-py/.venv/lib/python3.11/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(
/Users/badru.siddique/magento/learning/ai-enterprise-rag/ai-guardrails-py/.venv/lib/python3.11/site-packages/guardrails/validator_service/__init__.py:81: UserWarning: Could not obtain an event loop. Falling back to synchronous validation.
  warnings.warn(


Safe prompt accepted: Good morning, ChatGPT. Can you tell me a joke?
Sensitive-data prompt result: None


ERROR:opentelemetry.exporter.otlp.proto.http.trace_exporter:Failed to export span batch due to timeout, max retries or shutdown.
